# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdrayan001/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The playbook ranks pages for human review using the validated Random Forest score together with transparent content signals.

Priority actions are:

- `REFRESH` — high model priority plus meaningful visibility and content staleness.
- `CTR_REVIEW` — visible page with a lower-than-expected CTR for its position range.
- `PROTECT` — high-value visible page without a clear immediate refresh signal; review before making changes.
- `MONITOR` — useful visibility but weaker evidence for immediate action.
- `LOW_PRIORITY` — limited evidence for near-term intervention.

Each row receives a reason code so a reviewer can understand why it was ranked.

The ranking is decision-support only. It does not prove that an action will improve traffic, rankings, or CTR. Human review is required before any content change.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier


# ---------------------------------------------------------
# 1. Load starter data
# ---------------------------------------------------------

data_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv")
]

data_path = next(
    (p for p in data_paths if p.exists()),
    None
)

assert data_path is not None, "Dataset file not found."

df_playbook = pd.read_csv(data_path)

print("Dataset shape:", df_playbook.shape)


# ---------------------------------------------------------
# 2. Define current decline proxy
# ---------------------------------------------------------

df_playbook["declining_proxy"] = (
    df_playbook["trend_direction"] == "down"
).astype(int)


# ---------------------------------------------------------
# 3. Prepare the validated Week-5 feature set
# ---------------------------------------------------------

model_features = [
    "impressions_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update"
]

df_playbook["avg_position_missing"] = (
    df_playbook["avg_position"] == 0
).astype(int)

df_playbook.loc[
    df_playbook["avg_position"] == 0,
    "avg_position"
] = np.nan

model_features = model_features + [
    "avg_position_missing"
]


# ---------------------------------------------------------
# 4. Train final Random Forest on the available dataset
# ---------------------------------------------------------

X = df_playbook[model_features]
y = df_playbook["declining_proxy"]

final_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

final_model.fit(X, y)

df_playbook["model_score"] = final_model.predict_proba(X)[:, 1]


# ---------------------------------------------------------
# 5. Build transparent reason signals
# ---------------------------------------------------------

position_bins = [0, 3, 10, 20, np.inf]
position_labels = ["1-3", "4-10", "11-20", "21+"]

df_playbook["position_bucket"] = pd.cut(
    df_playbook["avg_position"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True
)

position_ctr_medians = (
    df_playbook
    .groupby("position_bucket", observed=True)["ctr"]
    .median()
)

df_playbook["bucket_ctr_median"] = (
    df_playbook["position_bucket"]
    .astype(object)
    .map(position_ctr_medians.to_dict())
    .astype(float)
)

df_playbook["ctr_opportunity"] = (
    df_playbook["bucket_ctr_median"].notna()
    & (
        df_playbook["ctr"]
        < df_playbook["bucket_ctr_median"]
    )
)

df_playbook["visible"] = (
    df_playbook["impressions_90d"] >= 500
)

df_playbook["stale"] = (
    df_playbook["days_since_last_update"] >= 91
)


# ---------------------------------------------------------
# 6. Create transparent action score
# ---------------------------------------------------------

df_playbook["action_score"] = (
    3 * df_playbook["ctr_opportunity"].astype(int)
    + 2 * (
        df_playbook["stale"]
        & df_playbook["visible"]
    ).astype(int)
    + df_playbook["visible"].astype(int)
)

# Combine validated model score with transparent signals.
df_playbook["priority_score"] = (
    df_playbook["model_score"]
    + 0.10 * (
        df_playbook["action_score"]
        / df_playbook["action_score"].max()
    )
)


# ---------------------------------------------------------
# 7. Assign action labels and reason codes
# ---------------------------------------------------------

def assign_action(row):
    if (
        row["model_score"] >= 0.70
        and row["visible"]
        and row["stale"]
        and row["ctr_opportunity"]
    ):
        return "REFRESH", "high_model_stale_ctr"

    if (
        row["model_score"] >= 0.70
        and row["visible"]
        and row["stale"]
    ):
        return "REFRESH", "high_model_stale_visible"

    if (
        row["model_score"] >= 0.70
        and row["visible"]
        and row["ctr_opportunity"]
    ):
        return "CTR_REVIEW", "high_model_ctr_gap"

    if (
        row["model_score"] >= 0.70
        and row["visible"]
    ):
        return "PROTECT", "high_model_high_visibility"

    if row["visible"]:
        return "MONITOR", "visible_lower_priority"

    return "LOW_PRIORITY", "limited_visibility"


actions = df_playbook.apply(
    assign_action,
    axis=1,
    result_type="expand"
)

actions.columns = [
    "action",
    "reason_code"
]

df_playbook[["action", "reason_code"]] = actions


# ---------------------------------------------------------
# 8. Rank final queue
# ---------------------------------------------------------

df_playbook = df_playbook.sort_values(
    [
        "priority_score",
        "impressions_90d"
    ],
    ascending=[False, False]
).reset_index(drop=True)

df_playbook["rank"] = np.arange(
    1,
    len(df_playbook) + 1
)


# ---------------------------------------------------------
# 9. Summary checks
# ---------------------------------------------------------

print("\nAction counts:")
print(df_playbook["action"].value_counts())

print("\nReason-code counts:")
print(df_playbook["reason_code"].value_counts())

print("\nTop 10 action queue:")

display(
    df_playbook[
        [
            "rank",
            "action",
            "reason_code",
            "priority_score",
            "model_score",
            "impressions_90d",
            "ctr",
            "avg_position",
            "content_age_days",
            "days_since_last_update"
        ]
    ].head(10).round(3)
)

assert len(df_playbook) == 30000
assert df_playbook["rank"].is_unique

print("\nRanked action queue check passed.")

Dataset shape: (30000, 44)

Action counts:
action
LOW_PRIORITY    13274
MONITOR         10959
PROTECT          2392
REFRESH          2389
CTR_REVIEW        986
Name: count, dtype: int64

Reason-code counts:
reason_code
limited_visibility            13274
visible_lower_priority        10959
high_model_high_visibility     2392
high_model_stale_visible       1556
high_model_ctr_gap              986
high_model_stale_ctr            833
Name: count, dtype: int64

Top 10 action queue:


,rank,action,reason_code,priority_score,model_score,impressions_90d,ctr,avg_position,content_age_days,days_since_last_update
0,1,REFRESH,high_model_stale_ctr,1.034,0.934,577,0.00,7.7,165,104
1,2,REFRESH,high_model_stale_ctr,1.033,0.933,1464,0.00,3.6,286,104
2,3,REFRESH,high_model_stale_ctr,1.032,0.932,646,0.00,14.5,165,104
3,4,REFRESH,high_model_stale_ctr,1.032,0.932,1150,0.00,16.3,148,104
4,5,REFRESH,high_model_stale_ctr,1.032,0.932,972,0.00,13.8,174,104
5,6,REFRESH,high_model_stale_ctr,1.031,0.931,1478,0.07,15.9,106,106
6,7,REFRESH,high_model_stale_ctr,1.030,0.930,817,0.00,14.7,174,104
7,8,REFRESH,high_model_stale_ctr,1.030,0.930,666,0.00,15.4,148,104
8,9,REFRESH,high_model_stale_ctr,1.028,0.928,1235,0.00,9.2,223,104
9,10,REFRESH,high_model_stale_ctr,1.027,0.927,618,0.00,3.6,131,104



Ranked action queue check passed.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This playbook is intended for SEO and content teams to prioritize pages for human review.

A reviewer can use the ranked queue to decide which pages may need refresh, CTR review, protection, monitoring, or lower-priority attention.

The playbook should not automatically publish, delete, rewrite, or redirect content. A high score is a prioritization signal, not proof that a page needs a specific action.

The model was validated using a client-grouped holdout, where Precision@50 was 0.720 compared with 0.620 for the Week-4 baseline. The random split produced 0.920, showing that validation design can materially change the measured result.

Important limits:
- The decline target is an observed current proxy, not a future outcome.
- The model does not prove causality.
- The model does not predict Google's algorithm.
- The recommendations may contain false positives and missed declining pages.
- Human review is required before taking action.
- Results may become stale as search behavior, content, and client data change.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Intended-use checks

intended_actions = {
    "REFRESH",
    "CTR_REVIEW",
    "PROTECT",
    "MONITOR",
    "LOW_PRIORITY"
}

actual_actions = set(df_playbook["action"].unique())

print("Intended actions:")
for action in sorted(intended_actions):
    print(" -", action)

print("\nActions produced by the playbook:")
for action in sorted(actual_actions):
    print(" -", action)

print("\nValidation evidence:")
print(f"Week-4 baseline Precision@50 : 0.620")
print(f"Grouped Random Forest P@50   : 0.720")
print(f"Random-split Random Forest P@50: 0.920")

print("\nUse limitation:")
print("Human review is required before content changes.")

assert actual_actions.issubset(intended_actions)
assert "trend_direction" not in model_features
assert "trend_pct" not in model_features
assert "client_id" not in model_features

print("\nIntended-use and limits check passed.")

Intended actions:
 - CTR_REVIEW
 - LOW_PRIORITY
 - MONITOR
 - PROTECT
 - REFRESH

Actions produced by the playbook:
 - CTR_REVIEW
 - LOW_PRIORITY
 - MONITOR
 - PROTECT
 - REFRESH

Validation evidence:
Week-4 baseline Precision@50 : 0.620
Grouped Random Forest P@50   : 0.720
Random-split Random Forest P@50: 0.920

Use limitation:
Human review is required before content changes.

Intended-use and limits check passed.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Every recommended action must be reviewed by a human before any content change.

Before acting, the reviewer should check:

- Whether the page still matches the search intent.
- Whether the page has useful, accurate, and up-to-date information.
- Whether the CTR signal may be explained by SERP layout, query mix, or the page type.
- Whether the page has important backlinks, conversions, or business value that are not represented in this model.
- Whether the recommended action is appropriate for the page's current position and visibility.
- Whether recent changes or unusual traffic patterns could explain the observed signals.

The following actions should never be automated by this playbook:

- Automatically deleting or pruning a page.
- Automatically rewriting or publishing content.
- Automatically changing titles, URLs, redirects, or canonical tags.
- Automatically deciding that a page is low quality.
- Automatically claiming that an action will recover traffic or rankings.

The model provides a ranked review queue. A human owns the final decision.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# Human-review and no-go checks
# ---------------------------------------------------------

no_go_actions = {
    "auto_delete",
    "auto_publish",
    "auto_rewrite",
    "auto_redirect",
    "auto_canonical_change",
    "auto_claim_recovery"
}

required_human_review = True

print("Human review required:", required_human_review)

print("\nNo-go actions:")
for action in sorted(no_go_actions):
    print(" -", action)

# The playbook must only produce review-oriented actions.
automated_actions = set(df_playbook["action"].unique())

allowed_actions = {
    "REFRESH",
    "CTR_REVIEW",
    "PROTECT",
    "MONITOR",
    "LOW_PRIORITY"
}

print("\nAutomated playbook actions:")
for action in sorted(automated_actions):
    print(" -", action)

assert required_human_review is True
assert automated_actions.issubset(allowed_actions)

# Confirm that no-go actions are not generated by the playbook.
assert automated_actions.isdisjoint(no_go_actions)

print("\nHuman-review and no-go check passed.")
print("The playbook does not perform irreversible content actions.")

Human review required: True

No-go actions:
 - auto_canonical_change
 - auto_claim_recovery
 - auto_delete
 - auto_publish
 - auto_redirect
 - auto_rewrite

Automated playbook actions:
 - CTR_REVIEW
 - LOW_PRIORITY
 - MONITOR
 - PROTECT
 - REFRESH

Human-review and no-go check passed.
The playbook does not perform irreversible content actions.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The playbook should be reviewed regularly because search behavior, content, and client data can change.

I would trigger a review or retraining when:

- The input data schema or feature definitions change.
- `avg_position` or other key features develop unusual missingness.
- The distribution of major features such as impressions, CTR, or average position shifts substantially.
- Precision@50 on a newly available labeled evaluation window falls meaningfully below the validated grouped result of 0.720.
- The ranking queue starts producing noticeably more false positives or missed declining pages during human review.
- The meaning or definition of the decline proxy changes.
- New clients, industries, or content types appear that are not well represented in the training data.

The model should not be retrained automatically just because a trigger occurs. A trigger starts an audit, after which the team can decide whether retraining is appropriate.

Until the model is revalidated, recommendations should be treated as lower-confidence decision-support signals.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---------------------------------------------------------
# Monitoring / retrain trigger checks
# ---------------------------------------------------------

monitoring_triggers = {
    "schema_or_feature_definition_change": True,
    "major_feature_distribution_shift": True,
    "increased_missingness": True,
    "precision_at_50_below_validated_0720": True,
    "human_review_error_increase": True,
    "target_definition_change": True,
    "new_unrepresented_clients_or_content_types": True
}

validated_precision_at_50 = 0.720

print("Validated grouped Precision@50:", validated_precision_at_50)

print("\nMonitoring / retrain triggers:")
for trigger in monitoring_triggers:
    print(" -", trigger)

print("\nRetraining policy:")
print("A trigger starts an audit; it does not automatically retrain the model.")

assert validated_precision_at_50 == 0.720
assert len(monitoring_triggers) >= 5

print("\nMonitoring and retrain trigger check passed.")

Validated grouped Precision@50: 0.72

Monitoring / retrain triggers:
 - schema_or_feature_definition_change
 - major_feature_distribution_shift
 - increased_missingness
 - precision_at_50_below_validated_0720
 - human_review_error_increase
 - target_definition_change
 - new_unrepresented_clients_or_content_types

Retraining policy:
A trigger starts an audit; it does not automatically retrain the model.

Monitoring and retrain trigger check passed.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The ranked action queue is exported to `work/outputs/` for reuse in the research paper.

The export contains the rank, recommended action, reason code, model score, priority score, and the main observable signals used to support the recommendation.

The current decline proxy is excluded from the paper-facing queue because it is an evaluation target, not an action input.

The exported queue is intended for human review and decision-support, not automatic content changes.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path

# ---------------------------------------------------------
# Export ranked action queue for the paper
# ---------------------------------------------------------

output_dir = Path("work/outputs")

if not output_dir.exists():
    output_dir = Path("../../work/outputs")

output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "content_action_playbook_queue.csv"

export_columns = [
    "rank",
    "action",
    "reason_code",
    "priority_score",
    "model_score",
    "impressions_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update"
]

queue_export = df_playbook[export_columns].copy()

queue_export.to_csv(
    output_path,
    index=False
)

print("Export path:", output_path)
print("Export rows:", len(queue_export))
print("Export columns:", len(queue_export.columns))

print("\nExport preview:")
display(queue_export.head(10).round(3))

# ---------------------------------------------------------
# Export checks
# ---------------------------------------------------------

assert output_path.exists()
assert len(queue_export) == 30000
assert queue_export["rank"].is_unique
assert queue_export["action"].notna().all()
assert queue_export["reason_code"].notna().all()

# Evaluation target must not be exported.
assert "declining_proxy" not in queue_export.columns
assert "trend_direction" not in queue_export.columns
assert "trend_pct" not in queue_export.columns

print("\nPaper export check passed.")
print("Ranked queue is ready for reuse.")

Export path: work\outputs\content_action_playbook_queue.csv
Export rows: 30000
Export columns: 11

Export preview:


,rank,action,reason_code,priority_score,model_score,impressions_90d,sessions_90d,ctr,avg_position,content_age_days,days_since_last_update
0,1,REFRESH,high_model_stale_ctr,1.034,0.934,577,5,0.00,7.7,165,104
1,2,REFRESH,high_model_stale_ctr,1.033,0.933,1464,10,0.00,3.6,286,104
2,3,REFRESH,high_model_stale_ctr,1.032,0.932,646,28,0.00,14.5,165,104
3,4,REFRESH,high_model_stale_ctr,1.032,0.932,1150,35,0.00,16.3,148,104
4,5,REFRESH,high_model_stale_ctr,1.032,0.932,972,27,0.00,13.8,174,104
5,6,REFRESH,high_model_stale_ctr,1.031,0.931,1478,2,0.07,15.9,106,106
6,7,REFRESH,high_model_stale_ctr,1.030,0.930,817,13,0.00,14.7,174,104
7,8,REFRESH,high_model_stale_ctr,1.030,0.930,666,47,0.00,15.4,148,104
8,9,REFRESH,high_model_stale_ctr,1.028,0.928,1235,34,0.00,9.2,223,104
9,10,REFRESH,high_model_stale_ctr,1.027,0.927,618,9,0.00,3.6,131,104



Paper export check passed.
Ranked queue is ready for reuse.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.